<span style="color:lightgreen">

# L4.1 Transcripción de portugués COVOST2

Ya que en este notebook no se realizan traducciones, no tiene cabida reporta COMET y BLEU

<span>

# ASR Baseline experiment using Whisper and CoVoST2 (Portuguese to English)

In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for ASR on the [CoVoST2 dataset](https://huggingface.co/datasets/facebook/covost2) (using the Portuguese-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing Word Error Rate, WER)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

import jiwer

model = whisper.load_model("base")

Load CoVoST2 speech dataset (Portuguese-English setup) from Hugging Face. Previously, audio data in the source language (version 4) must be downloaded from [Common Voice](https://commonvoice.mozilla.org/en/datasets)

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 9158
    })
    validation: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 3318
    })
    test: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 4023
    })
})


<p style="page-break-after:always;"></p>

Transcribe all the audio data using the Whisper (base) model. The ASR output is stored in hypotheses. At the same time, transcription references are stored into references and translation references into translations.

In [4]:
data = raw_datasets["test"]
hypotheses = []

for sample in tqdm(data["file"]):
    hypotheses.append((model.transcribe(sample, language=CONFIG.src_name))['text'])

  0%|          | 0/4023 [00:00<?, ?it/s]

<p style="page-break-after:always;"></p>

Transcription hypotheses, references and translations are stored into a Pandas dataframe. Show the two first hypotheses.

In [5]:
data_df = pd.DataFrame(dict(
    hypothesis=hypotheses, 
    reference=data["sentence"], 
    translation=data["translation"]
))
pd.set_option('display.max_colwidth', None)
data_df.head(2)

,hypothesis,reference,translation
0,e dia de ir emprestado as pessoas daudei.,Pedir dinheiro emprestado às pessoas da aldeia,Borrow money from people in the village
1,Tronca-vos.,Trancá-los,Lock them up


Hypotheses and references are normalized using the Whisper Basic text standardisation/normalization module

In [6]:
normalizer = BasicTextNormalizer()

data_df["hypothesis_clean"] = [normalizer(text) for text in data_df["hypothesis"]]
data_df["reference_clean"] = [normalizer(text) for text in data_df["reference"]]
data_df["translation_clean"] = [normalizer(text) for text in data_df["translation"]]
data_df.head(2)

,hypothesis,reference,translation,hypothesis_clean,reference_clean,translation_clean
0,e dia de ir emprestado as pessoas daudei.,Pedir dinheiro emprestado às pessoas da aldeia,Borrow money from people in the village,e dia de ir emprestado as pessoas daudei,pedir dinheiro emprestado às pessoas da aldeia,borrow money from people in the village
1,Tronca-vos.,Trancá-los,Lock them up,tronca vos,trancá los,lock them up


Finally, we compute the transcription WER using [JIWER](https://openai.com/index/whisper/) which is a simple and fast python package to evaluate ASR performance.

In [7]:

wer = jiwer.wer(list(data_df["reference_clean"]), list(data_df["hypothesis_clean"]))

print(f"WER: {wer * 100:.2f} %")

WER: 23.81 %


All the data is stored into a file using 'csv' format

In [8]:
data_df.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv'), encoding='utf-8')

# Ahora generamos datos para poder finetunear los modelos

In [9]:
data = raw_datasets["train"][:1000] # Cogemos 1000 datos para entrenamiento
hypotheses = []

for sample in tqdm(data["file"]):
    hypotheses.append((model.transcribe(sample, language=CONFIG.src_name))['text'])


data_df = pd.DataFrame(dict(
    hypothesis=hypotheses, 
    reference=data["sentence"], 
    translation=data["translation"]
))
# pd.set_option('display.max_colwidth', None)
# data_df.head(2)

normalizer = BasicTextNormalizer()

data_df["hypothesis_clean"] = [normalizer(text) for text in data_df["hypothesis"]]
data_df["reference_clean"] = [normalizer(text) for text in data_df["reference"]]
data_df["translation_clean"] = [normalizer(text) for text in data_df["translation"]]
# data_df.head(2)

data_df.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.1_ASR_Whisper_Baseline_Covost2_pt-en-train.csv'), encoding='utf-8')

  0%|          | 0/1000 [00:00<?, ?it/s]